<a href="https://colab.research.google.com/github/Muskan-Kandel/aes/blob/ayushma%2Fmistral-cot-implementation/AES_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#making a dummy data for testing

import pandas as pd

#dummy ds

df = pd.DataFrame({
    "essay" : [
        "Dear Editor, I believe local computers are beneficial because ...",
        "The internet has changed how students learn in many ways..."
    ],
    "content": [3,4],
    "language": [2.5,3.5],
    "adherence": [4,4],
    "narrativity":  [3,4],
    "holistic": [3,4]
})

df.head()

,essay,content,language,adherence,narrativity,holistic
0,"Dear Editor, I believe local computers are ben...",3,2.5,4,3,3
1,The internet has changed how students learn in...,4,3.5,4,4,4


In [2]:
!pip install -U bitsandbytes accelerate transformers
#for getting. the latest available version of bitsandbytes.
#run this only if the next block of code shows importerror, typically happens when session restarts so run each time with a new runtime

In [3]:
#torch -> for building loss functions and training
#transformers -> for mistraal/gpt type models

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

#4-bit configuration for memory efficiency

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name) #turns text to numbers that the model understands
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bnb_config,
                                             device_map="auto")

#this block takes a little longer to load, maybe like ~ 6-7 minutes


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [4]:
essay = df['essay'][0]
messages = [
    {
        "role" : "user",
        "content" : f"""

        Essay: {essay}

        Carefully analyze the essay.
        Then give scores (0-10) for:

        Content
        Language
        Adherence
        Narrativity
        Holistic

        Then provide reasoning for the scoring.

        Format should be strictly like this:

        Content: <score>
        Language: <score>
        Adherence: <score>
        Narrativity: <score>
        Holistic: <score>
        Reasoning: <text>

        """
    }
]

inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt"
)

inputs = {
    k: v.to(model.device) for k, v in inputs.items()
}

In [5]:
#generating a response

outputs = model.generate(
    **inputs,
    max_new_tokens = 300
)

result = tokenizer.decode(outputs[0], skip_special_tokens = True)
print(result)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] 
        
        Essay: Dear Editor, I believe local computers are beneficial because ...

        Carefully analyze the essay.
        Then give scores (0-10) for:

        Content
        Language
        Adherence
        Narrativity
        Holistic

        Then provide reasoning for the scoring.

        Format should be strictly like this:

        Content: <score>
        Language: <score>
        Adherence: <score>
        Narrativity: <score>
        Holistic: <score>
        Reasoning: <text>

         [/INST] Content: 8
The essay presents a clear and concise argument in favor of the use of local computers. The writer identifies the benefits of local computers without going off topic.

Language: 7
The language used in the essay is simple and easy to understand. However, there are some areas where the language could be improved to make the essay more engaging and persuasive.

Adherence: 9
The essay adheres to the format of a persuasive essay, with an introduction, bod

In [7]:
import re #regex module, used as a parser

scores = re.findall (r'(?:Content|Language|Adherence|Narrativity|Holistic):\s*(\d+)', result)
scores = [int(s) for s in scores]

print("extracted scores : " ,scores)

extracted scores :  [8, 7, 9, 5, 7]


In [10]:
import nbformat

#loading the notebook
with open("AES.ipynb", "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

#removing widget metadata
for cell in nb.cells:
    if "metadata" in cell and "widgets" in cell.metadata:
        cell.metadata.pop("widgets")

#saving fixed notebook
with open("AES_fixed.ipynb", "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Notebook fixed!")

#this code is run because colab notebooks added widget metadata and github's
# notebook renderer expects a state key inside each widget,
#so when its missing, github says invalid notebook

FileNotFoundError: [Errno 2] No such file or directory: 'AES.ipynb'